# detach-stop-gradient-trick — ex2: no_grad equivalent for the GAN D-step

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `detach-stop-gradient-trick`. Running the final beacon cell reports progress against the `GAN: detach stop-gradient trick` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: detach stop-gradient trick` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`detach-stop-gradient-trick`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "detach-stop-gradient-trick"
DD_SUBTOPIC = "GAN: detach stop-gradient trick"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `with torch.no_grad():` vs `.detach()` — quick refresher

Two ways to keep gradients from flowing into G during the D-step:

**`.detach()` (ex1):** `fake = G(z).detach()`. G's forward STILL builds the autograd graph; detach severs the link AFTER the fact. Cost: full forward graph allocation, then one node-level detach.

**`torch.no_grad()` (this drill):** `with torch.no_grad(): fake = G(z)`. G's forward DOES NOT build the autograd graph at all — no intermediate activation buffers, no edges. Cheaper.

Both produce a bit-identical `fake` tensor (same forward math) and leave D's gradient identical (D's path to the loss is the same). The difference is purely memory + compute on G's forward.

**Why `.detach()` is still common.** Makes the stop-gradient point EXPLICIT at the use site. `no_grad()` scopes the whole block — if you later add another op inside the `with`, it silently won't get a grad either. `.detach()` only severs the one tensor you name.

### Exercise 2 — no_grad equivalent for the GAN D-step

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the `torch.no_grad()` alternative to `.detach()` in the GAN D-step by implementing both and verifying identical fake values + identical D gradients + the autograd graph absence for the no_grad path.
> Keywords: no_grad, detach, gan, autograd-graph
> ```

**KCs targeted:** `no-grad-vs-detach-equivalence`, `no-grad-skips-graph-construction`

Implement `ex2_d_loss_with_no_grad(G, D, z, x_real)`. The cheaper alternative to ex1's `.detach()`:

1. Wrap G's forward in `with t.no_grad():` — `fake = G(z)` inside the block.
2. Compute `loss = (D(fake) - D(x_real)).mean()`.
3. `loss.backward()`.
4. Return `(loss.item(), fake.requires_grad)` — the second element is the test's way of probing that `fake` was built WITHOUT an autograd graph (under no_grad, the produced tensor has `requires_grad=False` and `grad_fn=None`).

The test confirms that:
- Loss value is identical to ex1's `.detach()` version (within 1e-5).
- D's parameter grads are identical to the `.detach()` version (modulo cumulative-grad ordering, which we control).
- G's parameter grads remain zero (same as `.detach()`).
- `fake.requires_grad is False` (the autograd-graph absence signature).

Input: `G`, `D` — modules; `z`, `x_real` — input tensors.
Output: `(loss_value: float, fake_requires_grad: bool)`.

In [ ]:
def ex2_d_loss_with_no_grad(G, D, z, x_real):
    with t.no_grad():
        fake = G(z)
    loss = (D(fake) - D(x_real)).mean()
    loss.backward()
    return loss.item(), fake.requires_grad


<details><summary>Solution</summary>

```python
def ex2_d_loss_with_no_grad(G, D, z, x_real):
    with t.no_grad():
        fake = G(z)
    loss = (D(fake) - D(x_real)).mean()
    loss.backward()
    return loss.item(), fake.requires_grad
```

**Memory savings.** `.detach()` runs G's forward inside the autograd-tracking machinery — every intermediate activation is kept alive in case the backward pass needs it. The detach only removes the OUTPUT edge; the internal tape is still built. `no_grad` skips graph construction entirely — no activation tape, no edges, just the value. For a deep G with many intermediate tensors, this is a measurable VRAM win during training.

**Why D's grads match exactly.** D's backward path only uses the VALUE of `fake`, not its grad_fn. Whether `fake` carries an autograd graph back to G's parameters or not is irrelevant to D's gradient computation — the forward through D is identical, and its backward only differentiates D's own weights w.r.t. the loss.

**When `.detach()` is still preferable.** When G's forward output is used in BOTH a no-grad-needed pass (D-step) and a grad-needed pass (G-step) in the same step. `no_grad` is a block scope; `.detach()` is per-tensor. Mix them: build the graph once for the G-step, detach a copy for the D-step.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()